In [2]:
import sys
sys.path.append("..")

from src.generate_scenarios import generate_scenarios

df = generate_scenarios(10000)

print("Number of scenarios:", len(df))
df.head()

Number of scenarios: 10000


,scenario_id,dirt_score,load_kg,fabric_type,water_availability_pct
0,SC02-00001,6.39,0.69,synthetic,31
1,SC02-00002,2.23,6.02,delicate,75
2,SC02-00003,4.22,0.72,cotton,29
3,SC02-00004,5.05,0.70,cotton,91
4,SC02-00005,6.50,4.59,cotton,57


In [2]:
import pandas as pd

In [3]:
# Check the generated data

print("Rows:", len(df))
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate scenario IDs:", df["scenario_id"].duplicated().sum())

print("\nMinimum values:")
print(df[["dirt_score", "load_kg", "water_availability_pct"]].min())

print("\nMaximum values:")
print(df[["dirt_score", "load_kg", "water_availability_pct"]].max())

Rows: 10000
Missing values:
scenario_id               0
dirt_score                0
load_kg                   0
fabric_type               0
water_availability_pct    0
dtype: int64

Duplicate scenario IDs: 0

Minimum values:
dirt_score                0.0
load_kg                   0.5
water_availability_pct    0.0
dtype: float64

Maximum values:
dirt_score                 10.0
load_kg                     8.0
water_availability_pct    100.0
dtype: float64


In [5]:
df.to_csv("../data/raw/scenarios.csv", index=False)

print("Saved successfully.")
print("File: data/raw/scenarios.csv")

Saved successfully.
File: data/raw/scenarios.csv


In [6]:
print("Fabric distribution:")
print(df["fabric_type"].value_counts())

print("\nPercentage distribution:")
print(df["fabric_type"].value_counts(normalize=True).round(3) * 100)

Fabric distribution:
fabric_type
delicate     2513
synthetic    2511
heavy        2508
cotton       2468
Name: count, dtype: int64

Percentage distribution:
fabric_type
delicate     25.1
synthetic    25.1
heavy        25.1
cotton       24.7
Name: proportion, dtype: float64


In [3]:
# Step 2 test scenarios

test_cases = pd.DataFrame([
    {
        "scenario_id": "TEST-NORMAL",
        "dirt_score": 5,
        "load_kg": 4.0,
        "fabric_type": "cotton",
        "water_availability_pct": 80
    },
    {
        "scenario_id": "TEST-BOUNDARY",
        "dirt_score": 0,
        "load_kg": 0.5,
        "fabric_type": "delicate",
        "water_availability_pct": 0
    },
    {
        "scenario_id": "TEST-STRESS",
        "dirt_score": 10,
        "load_kg": 8.0,
        "fabric_type": "heavy",
        "water_availability_pct": 100
    },
    {
        "scenario_id": "TEST-INVALID",
        "dirt_score": 11,
        "load_kg": 9.0,
        "fabric_type": "unknown",
        "water_availability_pct": 110
    }
])

test_cases

,scenario_id,dirt_score,load_kg,fabric_type,water_availability_pct
0,TEST-NORMAL,5,4.0,cotton,80
1,TEST-BOUNDARY,0,0.5,delicate,0
2,TEST-STRESS,10,8.0,heavy,100
3,TEST-INVALID,11,9.0,unknown,110


In [4]:
# Validate the test cases

def validate_test_case(row):
    errors = []

    if not 0 <= row["dirt_score"] <= 10:
        errors.append("Invalid dirt_score")

    if not 0.5 <= row["load_kg"] <= 8.0:
        errors.append("Invalid load_kg")

    if row["fabric_type"] not in ["delicate", "cotton", "synthetic", "heavy"]:
        errors.append("Invalid fabric_type")

    if not 0 <= row["water_availability_pct"] <= 100:
        errors.append("Invalid water_availability_pct")

    return "PASS" if not errors else "FAIL: " + ", ".join(errors)


test_cases["validation_result"] = test_cases.apply(validate_test_case, axis=1)

test_cases[["scenario_id", "validation_result"]]

,scenario_id,validation_result
0,TEST-NORMAL,PASS
1,TEST-BOUNDARY,PASS
2,TEST-STRESS,PASS
3,TEST-INVALID,"FAIL: Invalid dirt_score, Invalid load_kg, Inv..."
